# 03 · Label extraction and selected-image download

**Single responsibility:** create deterministic train/validation/test manifests and acquire only the real COCO images used

Run after the preceding numbered notebook unless the inputs already exist. Every generated artifact is written outside the notebook so this stage is reproducible.


In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/base.yaml').exists())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from vww_esp32.config import load_config, resolve_paths, seed_everything

config, ROOT = load_config(ROOT / 'configs/base.yaml')
paths = resolve_paths(config, ROOT)
seed_everything(config['project']['seed'])
ROOT


PosixPath('/Volumes/VM_SSD/Machine Learning/visual-wake-word-esp32')

In [2]:
from vww_esp32.data import build_manifest

annotation_dir = paths['raw'] / 'annotations'
manifest = build_manifest(
    annotation_dir / 'instances_train2017.json',
    annotation_dir / 'instances_val2017.json',
    paths['raw'] / 'images',
    min_area_fraction=config['data']['min_person_area_fraction'],
    validation_fraction=config['data']['validation_fraction'],
    max_samples=config['data']['max_samples'],
    balance_training_classes=config['data']['balance_training_classes'],
    seed=config['project']['seed'],
)
manifest.groupby(['split', 'label']).size().rename('images').to_frame()

/Volumes/VM_SSD/Machine Learning/visual-wake-word-esp32/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


images
split label        
test  0        1019
      1         981
train 0        6000
      1        6000
val   0        1011
      1         989

The training subset is balanced to improve optimization. Validation and test preserve their sampled COCO prevalence so operating metrics remain interpretable.

In [3]:
from vww_esp32.data import download_manifest_images

manifest = download_manifest_images(manifest, workers=config['data']['download_workers'], timeout=config['data']['download_timeout_seconds'])
failures = manifest.attrs.get('failures', [])
print(f'Downloaded: {manifest.downloaded.sum():,}/{len(manifest):,}; failures: {len(failures):,}')
failures[:5]

COCO images: 100%|██████████| 16000/16000 [48:52<00:00,  5.46it/s] 

Downloaded: 16,000/16,000; failures: 0


[]

In [4]:
interim_manifest = paths['interim'] / 'manifest.csv'
manifest.to_csv(interim_manifest, index=False)
assert manifest.image_id.is_unique
assert set(manifest.split) == {'train', 'val', 'test'}
assert set(manifest.label) <= {0, 1}
interim_manifest

PosixPath('/Volumes/VM_SSD/Machine Learning/visual-wake-word-esp32/data/interim/manifest.csv')